# 02 — Directed-Dyad Data (1946–2014)

Builds **25 directed-dyad files** (`dd_{cy}_{ud}.parquet`, `cy` ∈ 1–5, `ud` ∈ 1–5) that feed
the intervention and spatial-weights stages.

**Two-stage imputation strategy**

- *CY stage (notebook 01)*: 5 country-year imputed datasets propagate uncertainty in
  regime, GDP, and population.
- *UD stage (this notebook)*: for each CY dataset, miceforest produces 5 imputed copies
  of the undirected dyad frame.  Imputation is done before the UD→DD expansion so that
  symmetric features (trade, peace score, capital distance) have identical values in
  both directed rows of every dyad — i.e. $x_{AB} = x_{BA}$ by construction.

**Data sources**

| Variable group | Source | Coverage |
|---|---|---|
| Civil war onset, capabilities | COW Intra-State Wars v5.1, NMC v6 | through 2016 |
| Interstate wars | COW Inter-State War Data v4.0 | through 2007 |
| Alliances | COW Formal Alliances v4.1 | through 2012 (LOCF → 2014) |
| Peace quality | Diehl, Goertz & Gallegos Peace Data v2.01 | through 2015 |
| Territorial claims | ICOW v10.1 | through 2001 |
| Bilateral trade | COW Trade v4.0 | through 2014 |
| Dispute expectations | Carroll & Kenkel DOE v2.0 | through 2012 (LOCF → 2014) |
| Colonial history | COW Colonial History v1.0 | static |
| Contiguity | COW Direct Contiguity v3.2 | through 2016 |


In [1]:
import os
import sys
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat

sys.path.insert(0, str(Path.cwd().parent / "src"))
from shadow.data.ccode import make_cc, fix_ccode, cc_series

warnings.filterwarnings("ignore")

RAW     = Path("../data/raw")
INTERIM = Path("../data/interim")

# Output directory for the dd files.  Defaults to canonical; the lag1 rebuild
# (timing-contamination fix, 2026-07) sets SHADOW_DD_DIR=../data/interim/lag1
# so the rebuilt chain never clobbers canonical files before validation.
INTERIM_OUT = Path(os.environ.get("SHADOW_DD_DIR", "../data/interim"))
INTERIM_OUT.mkdir(parents=True, exist_ok=True)

N_IMP      = 5
YEAR_START = 1946
YEAR_END   = 2014

# Capital distance: map our analysis codes → codes present in capdist.csv.
# capdist was built for a post-2000 snapshot; some historical codes differ.
CAPDIST_RECODE = {
    "347": "345",   # Serbia/Montenegro → Yugoslavia (Belgrade same capital)
    "364": "365",   # USSR/Russia (our code 364 → capdist file uses 365)
    "529": "530",   # Ethiopia post-Eritrea → Ethiopia (Addis Ababa same)
    "769": "770",   # pre-Bangladesh Pakistan
    "818": "816",   # Unified Vietnam → North Vietnam (Hanoi)
}

print(f"Setup complete.  dd output → {INTERIM_OUT.resolve()}")

Setup complete.  dd output → /Users/rjc/portfolio/shadow/data/interim/lag1


## § 1 — Interstate Wars (COW v4.0)

Three undirected binary indicators per dyad-year:
- `ud_iswar_allied` — both in the same IS war, same side this year
- `ud_iswar_enemy` — both in the same IS war, opposite sides this year
- `ud_iswarin10yr` — fought each other in any IS war within the past 10 years

In [2]:
wars_raw = pd.read_csv(RAW / "cow/Inter-StateWarData_v4.0.csv")

records = []
for _, row in wars_raw.iterrows():
    cc   = make_cc(int(row["ccode"]))
    side = int(row["Side"])
    wn   = int(row["WarNum"])
    for sc, ec in [("StartYear1", "EndYear1"), ("StartYear2", "EndYear2")]:
        sy, ey = row[sc], row[ec]
        if pd.notna(sy) and sy > 0 and pd.notna(ey) and ey > 0:
            for yr in range(int(sy), int(ey) + 1):
                records.append({"WarNum": wn, "ccode": cc, "year": yr, "side": side})

war_py = pd.DataFrame(records).drop_duplicates()

wA = war_py.rename(columns={"ccode": "ccode_A", "side": "side_A"})
wB = war_py.rename(columns={"ccode": "ccode_B", "side": "side_B"})
war_pairs = (
    pd.merge(wA, wB, on=["WarNum", "year"])
    .query("ccode_A < ccode_B")
    .assign(
        allied=lambda d: (d["side_A"] == d["side_B"]).astype(int),
        enemy =lambda d: (d["side_A"] != d["side_B"]).astype(int),
    )
)

war_ud = (
    war_pairs
    .groupby(["ccode_A", "ccode_B", "year"], as_index=False)
    .agg(ud_iswar_allied=("allied", "max"), ud_iswar_enemy=("enemy", "max"))
)

# Enemy-in-past-10-years: for each enemy dyad-year, flag the following 10 years
enemy_hist = war_ud.loc[war_ud["ud_iswar_enemy"] == 1, ["ccode_A", "ccode_B", "year"]]
in10_rows = []
for _, r in enemy_hist.iterrows():
    for fy in range(r["year"] + 1, r["year"] + 11):
        in10_rows.append({"ccode_A": r["ccode_A"], "ccode_B": r["ccode_B"], "year": fy})

in10 = pd.DataFrame(in10_rows).drop_duplicates().assign(ud_iswarin10yr=1)
war_ud = war_ud.merge(in10, on=["ccode_A", "ccode_B", "year"], how="left")
war_ud["ud_iswarin10yr"] = war_ud["ud_iswarin10yr"].fillna(0).astype(int)

print(f"IS war UD table: {len(war_ud):,} rows")

IS war UD table: 2,244 rows


## § 2 — Capital Distances (static)

In [3]:
capdist_raw = pd.read_csv(RAW / "cow/capdist.csv")
capdist_raw["ca"] = capdist_raw["numa"].apply(lambda x: make_cc(int(x)))
capdist_raw["cb"] = capdist_raw["numb"].apply(lambda x: make_cc(int(x)))

# Build symmetric lookup: (min, max) → kmdist
capdist_lookup: dict[tuple, float] = {}
for _, row in capdist_raw.iterrows():
    ca, cb, d = row["ca"], row["cb"], row["kmdist"]
    capdist_lookup[(min(ca, cb), max(ca, cb))] = d

# Add alias entries so analysis codes (e.g. 347) resolve via historical codes (345)
extra: dict[tuple, float] = {}
for anal_code, cap_code in CAPDIST_RECODE.items():
    for (c1, c2), d in capdist_lookup.items():
        if c1 == cap_code:
            extra[(min(anal_code, c2), max(anal_code, c2))] = d
        elif c2 == cap_code:
            extra[(min(c1, anal_code), max(c1, anal_code))] = d
capdist_lookup.update(extra)

def get_capdist(cA: str, cB: str) -> float:
    return capdist_lookup.get((min(cA, cB), max(cA, cB)), np.nan)

print(f"capdist lookup: {len(capdist_lookup):,} entries")

capdist lookup: 21,308 entries


## § 3 — Contiguity (COW Direct Contiguity v3.2)

In [4]:
# conttype: 1=land/river, 2=water<12mi, 3=12-24mi, 4=24-150mi, 5=150-400mi
# begin/end in YYYYMM → extract year by integer-dividing by 100
cont_raw = pd.read_csv(RAW / "cow/DirectContiguity320/contdir.csv")
cont_raw["begin_yr"] = (cont_raw["begin"] // 100).astype(int)
cont_raw["end_yr"]   = (cont_raw["end"]   // 100).astype(int)
cont_raw["cA"] = cont_raw[["statelno", "statehno"]].min(axis=1).apply(make_cc)
cont_raw["cB"] = cont_raw[["statelno", "statehno"]].max(axis=1).apply(make_cc)

contig_rows = []
for _, row in cont_raw.iterrows():
    y0 = max(row["begin_yr"], 1940)
    y1 = min(row["end_yr"], YEAR_END)
    if y0 > y1:
        continue
    for yr in range(y0, y1 + 1):
        contig_rows.append({"ccode_A": row["cA"], "ccode_B": row["cB"],
                            "year": yr, "conttype": int(row["conttype"])})

contig_df = pd.DataFrame(contig_rows)
contig_ud = (
    contig_df
    .groupby(["ccode_A", "ccode_B", "year"], as_index=False)["conttype"]
    .min()
    .rename(columns={"conttype": "ud_conttype"})
)

print(f"Contiguity UD table: {len(contig_ud):,} rows")
print(contig_ud["ud_conttype"].value_counts().sort_index())

Contiguity UD table: 28,319 rows
ud_conttype
1    16383
2      754
3      679
4     4024
5     6479
Name: count, dtype: int64


## § 4 — Formal Alliances (COW v4.1, through 2012)

COW Formal Alliances v4.1 uses a directed-yearly format where each
dyad-year row represents the alliance commitment of ccode1 toward ccode2.
Variables: `defense`, `neutrality`, `nonaggression`, `entente`.

We standardize to undirected dyads (min/max ccode ordering) and take the
maximum across both directed rows.  2013–2014 gap filled by LOCF in the
main loop.

In [5]:
ally_raw = pd.read_csv(
    RAW / "cow/FormalAlliances/version4.1_csv/alliance_v4.1_by_directed_yearly.csv"
)

ally_raw["ccode_A"] = ally_raw[["ccode1", "ccode2"]].min(axis=1).apply(make_cc)
ally_raw["ccode_B"] = ally_raw[["ccode1", "ccode2"]].max(axis=1).apply(make_cc)

ALLY_COLS = ["defense", "neutrality", "nonaggression", "entente"]
ally_agg  = {f"ud_{c}": (c, "max") for c in ALLY_COLS}

ally_ud = (
    ally_raw
    .groupby(["ccode_A", "ccode_B", "year"], as_index=False)
    .agg(**ally_agg)
)

n_any = (ally_ud[[f"ud_{c}" for c in ALLY_COLS]].max(axis=1) > 0).sum()
print(f"Alliance UD table: {len(ally_ud):,} rows")
print(f"Year range: {ally_ud['year'].min()}–{ally_ud['year'].max()}")
print(f"Dyad-years with any alliance: {n_any:,}")

Alliance UD table: 60,392 rows
Year range: 1816–2012
Dyad-years with any alliance: 60,392


## § 5 — Peace Data (Diehl, Goertz & Gallegos v2.01)

Replaces the Thompson-Dreyer rivalry typology with the continuous Peace score.

| Score | Meaning |
|---|---|
| 0.00 | Hostile / active rivalry |
| 0.25 | Low-level conflict |
| 0.50 | Negative peace (cold peace) |
| 0.75 | Positive peace |
| 1.00 = | Stable peace |

File format: `AAABBB, YYYYMMDD-YYYYMMDD, score, ...` (variable intervals per dyad).  
Dyads not in the dataset are left as NaN (genuinely unmeasured, not assumed peaceful).  
Transition years with two scores take the minimum (more hostile) value.

In [6]:
def parse_peace_data(path) -> pd.DataFrame:
    """Parse the Peace data file into a (ccode_A, ccode_B, year, ud_peace) frame."""
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = [p.strip() for p in line.split(',')]
            dyad_id = parts[0]
            if len(dyad_id) != 6:
                continue
            cA, cB = dyad_id[:3], dyad_id[3:]
            i = 1
            while i + 1 <= len(parts) - 1:
                dr, sc = parts[i], parts[i + 1]
                i += 2
                if not dr or not sc:
                    continue
                try:
                    sy = int(dr[:4])
                    ey = int(dr[9:13])
                except (ValueError, IndexError):
                    continue
                if ey >= 9999:
                    ey = YEAR_END
                try:
                    score = float(sc)
                except ValueError:
                    continue
                if score < 0:          # -9 sentinel = missing
                    continue
                for yr in range(max(sy, YEAR_START), min(ey, YEAR_END) + 1):
                    rows.append({'ccode_A': cA, 'ccode_B': cB,
                                 'year': yr, 'ud_peace': score})
    df = pd.DataFrame(rows)
    # Transition years may have two scores; take the minimum (most hostile)
    df = (
        df.groupby(['ccode_A', 'ccode_B', 'year'], as_index=False)['ud_peace']
        .min()
    )
    return df


peace_ud = parse_peace_data(RAW / 'peacedata/peacedatav2.01.csv')

print(f'Peace data: {len(peace_ud):,} dyad-year rows')
print(f'Unique dyads: {peace_ud[["ccode_A","ccode_B"]].drop_duplicates().shape[0]:,}')
print(f'Year range: {peace_ud["year"].min()}\u2013{peace_ud["year"].max()}')
print('Score distribution:')
print(peace_ud['ud_peace'].value_counts().sort_index())


Peace data: 73,348 dyad-year rows
Unique dyads: 2,508
Year range: 1946–2014
Score distribution:
ud_peace
0.00     3682
0.25     2914
0.50    55735
0.75     5445
1.00     5572
Name: count, dtype: int64


## § 6 — ICOW Territorial Claims (v10.1, through 2001)

In [7]:
icow_raw, _ = pyreadstat.read_dta(RAW / "icow/ICOWprovyr101.dta")
icow_raw = icow_raw.dropna(subset=["chal", "tgt", "year"])

icow_raw["ccode_A"] = icow_raw[["chal", "tgt"]].min(axis=1).apply(
    lambda x: make_cc(int(x))
)
icow_raw["ccode_B"] = icow_raw[["chal", "tgt"]].max(axis=1).apply(
    lambda x: make_cc(int(x))
)
icow_raw["year"] = icow_raw["year"].astype(int)

icow_ud = (
    icow_raw
    .groupby(["ccode_A", "ccode_B", "year"], as_index=False)
    .agg(
        ud_icow_nclaims=("claim", "count"),
        ud_icow_maxsal=("icowsal", "max"),
    )
)

print(f"ICOW UD table: {len(icow_ud):,} rows")

ICOW UD table: 10,525 rows


## § 7 — Dyadic Trade (COW v4.0, 1870–2014)

`flow1` = ccode1 imports from ccode2; `flow2` = ccode2 imports from ccode1.
Since the trade file always has ccode1 < ccode2, for our undirected dyad
(A = smaller ccode): `ud_A_biImports = flow1`, `ud_B_biImports = flow2`.

We use `smoothflow1`/`smoothflow2` (Gleditsch's interpolated series) to reduce
data-quality noise; fall back to raw `flow1`/`flow2` where smooth is missing.

In [8]:
with zipfile.ZipFile(RAW / "cow/COW_Trade_4.0.zip") as z:
    with z.open("COW_Trade_4.0/Dyadic_COW_4.0.csv") as f:
        trade_raw = pd.read_csv(
            f,
            usecols=["ccode1", "ccode2", "year",
                     "flow1", "flow2", "smoothflow1", "smoothflow2"]
        )

trade_raw["ccode_A"] = trade_raw["ccode1"].apply(make_cc)
trade_raw["ccode_B"] = trade_raw["ccode2"].apply(make_cc)

# -9 = COW sentinel for unknown/missing
for col in ["flow1", "flow2", "smoothflow1", "smoothflow2"]:
    trade_raw[col] = trade_raw[col].replace(-9, np.nan)

# Prefer smoothed; fall back to raw
trade_raw["ud_A_biImports"] = trade_raw["smoothflow1"].fillna(trade_raw["flow1"])
trade_raw["ud_B_biImports"] = trade_raw["smoothflow2"].fillna(trade_raw["flow2"])

trade_ud = (
    trade_raw[["ccode_A", "ccode_B", "year", "ud_A_biImports", "ud_B_biImports"]]
    .query("@YEAR_START <= year <= @YEAR_END")
    .copy()
)

print(f"Trade UD: {len(trade_ud):,} rows, {trade_ud['year'].min()}–{trade_ud['year'].max()}")
print(f"A_biImports coverage: {trade_ud['ud_A_biImports'].notna().mean():.1%}")

Trade UD: 798,157 rows, 1946–2014
A_biImports coverage: 77.8%


## § 8 — DOE Scores (Carroll & Kenkel v2.0, through 2012)

Directed dyad-year predictions from the Dispute Outcome Expectations model.
Variables: `pr_win_a`, `pr_stalemate`, `pr_win_b`.
Gap for 2013–2014 filled by LOCF within each dyad in the main loop.

In [9]:
doe_raw = pd.read_csv(RAW / "doe/doe-dir-dyad-2.0.csv")

doe_raw["ccode_A"] = doe_raw["ccode_a"].apply(make_cc)
doe_raw["ccode_B"] = doe_raw["ccode_b"].apply(make_cc)
doe_raw = doe_raw.dropna(subset=["ccode_A", "ccode_B"])

doe_ud = (
    doe_raw[["ccode_A", "ccode_B", "year",
              "pr_win_a", "pr_stalemate", "pr_win_b"]]
    .query("@YEAR_START <= year <= @YEAR_END")
    .rename(columns={
        "pr_win_a":    "doe_pr_win_A",
        "pr_stalemate": "doe_pr_stalemate",
        "pr_win_b":    "doe_pr_win_B",
    })
    .copy()
)

# DOE is already directed (ccode_A vs ccode_B order matters)
# Keep as-is; during DD expansion we swap A/B and also swap pr_win_A/pr_win_B
print(f"DOE directed table: {len(doe_ud):,} rows, {doe_ud['year'].min()}–{doe_ud['year'].max()}")
print(f"pr_win_A coverage: {doe_ud['doe_pr_win_A'].notna().mean():.1%}")

DOE directed table: 1,520,658 rows, 1946–2012
pr_win_A coverage: 100.0%


## § 9 — Colonial History (COW coldata100)

In [10]:
col_raw = pd.read_csv(RAW / "cow/coldata100.csv")

col_pairs = (
    col_raw[col_raw["IndFrom"].notna() & (col_raw["IndFrom"] > 0)]
    [["State", "IndFrom"]].copy()
)
col_pairs["colony"]    = col_pairs["State"].apply(make_cc)
col_pairs["colonizer"] = col_pairs["IndFrom"].apply(lambda x: make_cc(int(x)))
col_pairs = col_pairs.dropna(subset=["colony", "colonizer"])

# (colony, colonizer) lookup
colonial_set: set[tuple] = set(zip(col_pairs["colony"], col_pairs["colonizer"]))

# Shared-colonizer pairs
shared_colonizer_set: set[tuple] = set()
for _, grp in col_pairs.groupby("colonizer")["colony"]:
    colonies = list(grp)
    for i, c1 in enumerate(colonies):
        for c2 in colonies[i + 1:]:
            shared_colonizer_set.add((min(c1, c2), max(c1, c2)))

print(f"Colonial pairs: {len(colonial_set)} (colony → colonizer)")
print(f"Shared-colonizer pairs: {len(shared_colonizer_set)}")

Colonial pairs: 178 (colony → colonizer)
Shared-colonizer pairs: 2273


## § 10 — Group Labels (from CY base file)

In [11]:
GROUP_COLS = [
    "first_eth_grp", "second_eth_grp",
    "first_lin_grp", "second_lin_grp",
    "first_rel_grp", "second_rel_grp",
]

# other_ongoing_wars rides along here: it is complete by construction (built
# from the COW war file in nb 01), so it never entered the CY imputation and
# is not in the cy_imputed_* files.  Merged onto the CY frame by cyear inside
# build_ud, exactly like the group labels.
cy_groups = pd.read_parquet(
    INTERIM / "country_year.parquet",
    columns=["cyear"] + GROUP_COLS + ["other_ongoing_wars"],
)

print(f"Group labels: {len(cy_groups):,} rows")
for col in GROUP_COLS:
    print(f"  {col}: {cy_groups[col].notna().mean():.1%}")
print(f"  other_ongoing_wars: complete, mean {cy_groups['other_ongoing_wars'].mean():.3f}")

Group labels: 9,280 rows
  first_eth_grp: 96.1%
  second_eth_grp: 83.4%
  first_lin_grp: 94.5%
  second_lin_grp: 72.5%
  first_rel_grp: 96.4%
  second_rel_grp: 79.4%
  other_ongoing_wars: complete, mean 0.051


## § 10b — IGO Shared Membership (COW IGO v3, through 2014)

Count of intergovernmental organizations in which **both** states hold full membership,
from the COW Dyadic IGO Dataset v3 (Pevehouse et al. 2020).

In the COW dyadic IGO encoding, a column value of **1** means both states in the dyad are
full members of that IGO.  The shared count is therefore the row-sum of (value == 1) across
all IGO columns.  Dyad-years absent from the file receive 0 (no shared memberships).

Shared IGO membership captures **institutionalised cooperation** — common organisational
environments create contact, shared norms, and monitoring mechanisms that can lower the
political cost of a coordinated intervention response (Pevehouse et al. 2004).

In [12]:
igo_raw = pd.read_csv(RAW / "cow/IGO/dyadic_formatv3.csv", low_memory=False)

meta_cols = ["ccode1", "country1", "ccode2", "country2", "year", "state"]
igo_val_cols = [c for c in igo_raw.columns if c not in meta_cols]

# Value == 1: both states are full members of this IGO
igo_raw["igo_shared"] = (igo_raw[igo_val_cols] == 1).sum(axis=1)

igo_raw["ccode_A"] = igo_raw["ccode1"].apply(make_cc)
igo_raw["ccode_B"] = igo_raw["ccode2"].apply(make_cc)

igo_ud = (
    igo_raw[["ccode_A", "ccode_B", "year", "igo_shared"]]
    .query("@YEAR_START <= year <= @YEAR_END")
    .copy()
)

print(f"IGO UD table: {len(igo_ud):,} rows")
print(f"Year range:   {igo_ud['year'].min()}–{igo_ud['year'].max()}")
print(f"Mean shared memberships: {igo_ud['igo_shared'].mean():.1f}")
print(f"Max shared memberships:  {igo_ud['igo_shared'].max()}")

IGO UD table: 797,509 rows
Year range:   1946–2014
Mean shared memberships: 24.9
Max shared memberships:  106


## § 11 — Helper Functions

### Timing policy: the t−1 information set (2026-07 rebuild)

Stage 1 estimates the probability that each potential intervener would enter **a war in A
at t**.  The war's *existence* is supplied by the game tree (and by the training universe,
which consists entirely of onset dyad-years) — it must not also be encoded in the features.
Every feature should therefore be **measurable at the challenger's decision point**, i.e.
from the information set at the end of t−1.  Concretely:

- **Shifted to t−1** (`TV_LAG`, both `_A` and `_B` sides via the pre-cross-join shift):
  all time-varying country-year covariates — regime (`polity2`, `instab`,
  `v2x_polyarchy`), economy (`lgdp`, `lpop`, `oil`; their `_lag` twins become t−2 depth),
  ethnic exclusion (`eth_excl_frac`), capabilities (`irst`, `milex`, `milper`, `pec`,
  `upop`, `cinc`), and UN voting (`ideal_point`).  The spatial-W matrices in notebook 04
  read `polity2_B` / `ideal_point_B` off these rows and inherit the shift automatically.
- **Already decision-point-timed (not shifted)**: `other_ongoing_wars` (wars ongoing at t
  excluding any phase that begins at t — replaces `ongoing_wars`, which counted the
  contested war itself and was ≥1 on every onset year by construction), `prior_war`
  (= ongoing(t−1) > 0), `recent_int` (window [t−5, t−1]).
- **Fixed or deterministic (not shifted)**: terrain, colonial history, fractionalization,
  regions, `nwstate` (state age), `major_power` / `is_P5` (institutional status known at
  the start of t), `year` / `cold_war` (the decision year indexes the row; not a leak).
- **Genuinely dyadic interstate variables stay at t by design** (`ud_peace`, trade,
  alliances, capital distance, DOE, `igo_shared`, ICOW, interstate-war history): they
  describe the A–B interstate relationship, not the host's civil-war state, and several
  are LOCF-extended slow-movers.  Documented as onset-orthogonal.

Boundary handling for the shift: the imputed panel starts in 1946, so `shift(1)` is
backstopped first by the **raw t−1 value** from `country_year.parquet` (which extends back
to 1940), then — for panel-entry years with no prior observation at all (new states;
sources that begin in 1946) — by the country's **own year-t value** (a documented, mild
look-ahead affecting a small set of rows).  A hard assertion guarantees the shifted
columns are complete, preserving the UD-imputation contract below.

In [13]:

# CY columns to carry into the dyad frame (for both A and B sides).
# NOTE: 'regan_period' is excluded here; it is added as a row-level column
# (year <= 1999) after the cross-join, avoiding redundant _A/_B duplicates.
# 'ongoing_wars' was REMOVED in the 2026-07 timing rebuild: it counted the
# war that begins at t (>= 1 on every onset year by construction), leaking
# the outcome into the prediction grid.  Its decision-point-timed
# replacement, other_ongoing_wars, is merged from country_year.parquet in
# build_ud (see the timing-policy note above).
CY_KEEP = [
    "ccode", "year", "cyear", "cname",
    "onset", "prior_war",
    "polity2", "instab", "v2x_polyarchy",
    "lgdp", "lgdp_lag", "lpop", "lpop_lag",
    "oil", "lmtnest", "ncontig", "nwstate",
    "colbrit", "colfra",
    "ethfrac", "relfrac", "muslim", "eth_excl_frac",
    "irst", "milex", "milper", "pec", "upop", "cinc",
    # § 6 additions: state power, voting position, intervention history
    "major_power", "is_P5", "ideal_point", "recent_int",
    "reg_asia", "reg_eeurope", "reg_latinAmerica", "reg_mena", "reg_western",
]

# Time-varying CY covariates shifted to t-1 (see timing-policy markdown).
TV_LAG = [
    "polity2", "instab", "v2x_polyarchy",
    "lgdp", "lgdp_lag", "lpop", "lpop_lag",
    "oil", "eth_excl_frac",
    "irst", "milex", "milper", "pec", "upop", "cinc",
    "ideal_point",
]

# Raw (unimputed) CY panel keyed to t+1: the boundary backstop for the shift.
# country_year.parquet extends back to 1940, so most states' 1946 rows can
# take their true 1945 values here rather than a carry-back.
cy_prev_raw = pd.read_parquet(
    INTERIM / "country_year.parquet", columns=["ccode", "year"] + TV_LAG
)
cy_prev_raw["ccode"] = cy_prev_raw["ccode"].astype(str)
cy_prev_raw["year"] = cy_prev_raw["year"] + 1


def build_ud(imp_path: Path) -> pd.DataFrame:
    """Cross-join one CY imputation file to build the undirected dyad frame."""
    cy = pd.read_parquet(imp_path)[CY_KEEP]

    # Group labels + other_ongoing_wars are not in the imputed files; merge
    # from base CY
    cy = cy.merge(cy_groups, on="cyear", how="left")
    assert cy["other_ongoing_wars"].notna().all()

    # ── t-1 information-set shift (timing-policy markdown above) ─────────
    cy = cy.sort_values(["ccode", "year"]).reset_index(drop=True)
    lagged = cy.groupby("ccode")[TV_LAG].shift(1)
    prev = cy[["ccode", "year"]].merge(cy_prev_raw, on=["ccode", "year"], how="left")
    for c in TV_LAG:
        lagged[c] = lagged[c].fillna(prev[c])   # boundary: raw value at t-1
        lagged[c] = lagged[c].fillna(cy[c])     # panel entry: own-t carry-back
    n_carryback = int(lagged.isna().sum().sum())  # before final fill: sanity only
    cy[TV_LAG] = lagged
    assert cy[TV_LAG].notna().all().all(), \
        "t-1 shift left NaNs (violates the UD-imputation completeness contract)"

    # Cross-join year-by-year to keep memory bounded
    chunks = []
    for yr, grp in cy.groupby("year"):
        cross = grp.merge(grp, on="year", suffixes=("_A", "_B"))
        cross = cross[cross["ccode_A"] < cross["ccode_B"]].copy()
        chunks.append(cross)

    ud = pd.concat(chunks, ignore_index=True)

    # Add row-level flag (same for all states in a given year)
    ud["regan_period"] = ud["year"] <= 1999
    ud["udyear"] = (
        ud["ccode_A"] + "_" + ud["ccode_B"] + "_" + ud["year"].astype(str)
    )
    return ud


def merge_ud_vars(ud: pd.DataFrame) -> pd.DataFrame:
    """Merge all precomputed UD auxiliary tables onto the dyad frame."""
    key = ["ccode_A", "ccode_B", "year"]

    ud = ud.merge(war_ud,    on=key, how="left")
    ud = ud.merge(contig_ud, on=key, how="left")
    ud = ud.merge(ally_ud,   on=key, how="left")
    ud = ud.merge(peace_ud,  on=key, how="left")
    ud = ud.merge(icow_ud,   on=key, how="left")
    ud = ud.merge(trade_ud,  on=key, how="left")
    ud = ud.merge(igo_ud,    on=key, how="left")   # § 10b: IGO shared membership

    # DOE is directed: ccode_A/ccode_B ordering in doe_ud matches the UD frame
    ud = ud.merge(doe_ud, on=key, how="left")

    # ── Absent-means-zero variables ───────────────────────────────────────────
    zero_fill = [
        "ud_iswar_allied", "ud_iswar_enemy", "ud_iswarin10yr",
        "ud_defense", "ud_neutrality", "ud_nonaggression", "ud_entente",
        "ud_icow_nclaims", "ud_icow_maxsal",
        "igo_shared",    # absent dyad-year = no shared memberships
    ]
    for col in zero_fill:
        if col in ud.columns:
            ud[col] = ud[col].fillna(0)

    # Contiguity: absent = 6 (not contiguous)
    ud["ud_conttype"] = ud["ud_conttype"].fillna(6)

    # ── LOCF for time-limited sources ─────────────────────────────────────────
    locf_cols = [
        "ud_A_biImports", "ud_B_biImports",
        "ud_defense", "ud_neutrality", "ud_nonaggression", "ud_entente",
        "doe_pr_win_A", "doe_pr_stalemate", "doe_pr_win_B",
    ]
    ud = ud.sort_values(["ccode_A", "ccode_B", "year"])
    for col in locf_cols:
        if col in ud.columns:
            ud[col] = ud.groupby(["ccode_A", "ccode_B"])[col].transform("ffill")

    # ── Static lookups ────────────────────────────────────────────────────────
    ud["ud_log_capdist"] = ud.apply(
        lambda r: np.log1p(get_capdist(r["ccode_A"], r["ccode_B"])), axis=1
    )

    # ── Group-overlap dummies ─────────────────────────────────────────────────
    for stem in ["eth", "lin", "rel"]:
        fcol = f"first_{stem}_grp"
        if f"{fcol}_A" in ud.columns:
            ud[f"ud_sameFirst{stem.capitalize()}"] = (
                ud[f"{fcol}_A"].notna()
                & ud[f"{fcol}_B"].notna()
                & (ud[f"{fcol}_A"] == ud[f"{fcol}_B"])
            ).astype(int)

    return ud


def swap_ab(df: pd.DataFrame) -> pd.DataFrame:
    """
    Swap the A and B sides of a dyad frame to produce the reverse-directed row.

    Column naming conventions:
      - Suffix _A / _B  (e.g. ccode_A, onset_A, doe_pr_win_A)  →  swap suffix
      - Prefix ud_A_ / ud_B_  (e.g. ud_A_biImports)            →  swap prefix
      - All other columns (symmetric)                           →  unchanged
    """
    rename = {}
    for col in df.columns:
        if col.startswith("ud_A_"):
            rename[col] = "ud_B_" + col[5:]
        elif col.startswith("ud_B_"):
            rename[col] = "ud_A_" + col[5:]
        elif col.endswith("_A"):
            rename[col] = col[:-2] + "_B"
        elif col.endswith("_B"):
            rename[col] = col[:-2] + "_A"
    return df.rename(columns=rename)


def ud_to_dd(ud: pd.DataFrame) -> pd.DataFrame:
    """Expand undirected dyad frame to directed by duplicating with A/B swapped."""
    row1 = ud.copy()
    row2 = swap_ab(ud.copy())

    dd = pd.concat([row1, row2], ignore_index=True)
    dd["ddyear"] = (
        dd["ccode_A"] + "_" + dd["ccode_B"] + "_" + dd["year"].astype(str)
    )

    # Directed capability ratio (sign inverts correctly after A/B swap)
    eps = 1e-10
    dd["log_capratio"] = np.log(
        (dd["cinc_A"] + eps) / (dd["cinc_B"] + eps)
    )

    # Directed colonial history flags
    dd["A_wasColOf_B"] = dd.apply(
        lambda r: int((r["ccode_A"], r["ccode_B"]) in colonial_set), axis=1
    )
    dd["B_wasColOf_A"] = dd.apply(
        lambda r: int((r["ccode_B"], r["ccode_A"]) in colonial_set), axis=1
    )
    dd["ud_sharedColonizer"] = dd.apply(
        lambda r: int(
            (min(r["ccode_A"], r["ccode_B"]),
             max(r["ccode_A"], r["ccode_B"])) in shared_colonizer_set
        ),
        axis=1,
    )

    # UN ideal point distance (symmetric; NaN where either side was imputed to NaN)
    dd["ideal_point_distance"] = (
        dd["ideal_point_A"] - dd["ideal_point_B"]
    ).abs()

    return dd


print("Helper functions defined.")


Helper functions defined.


In [14]:

# ── UD-level imputation ───────────────────────────────────────────────────────
# Imputation is performed on the *undirected* dyad frame (one row per AB pair)
# so that symmetric quantities (trade, peace, distance) are imputed once and
# then flipped deterministically by swap_ab.  Imputing in the DD frame would
# allow x_AB ≠ x_BA for the same underlying measurement.

import miceforest as mf
from scipy.stats import kstest as _kstest


def _find_asinh_theta(series: pd.Series, n_points: int = 40) -> float:
    """Find theta that minimises KS(asinh(theta * x), Normal(0,1))."""
    s = series.dropna()
    if len(s) < 10:
        return 1.0
    best_theta, best_ks = 1.0, np.inf
    for exp in np.linspace(-10, 10, n_points):
        theta = 2.0 ** exp
        t = np.arcsinh(theta * s)
        t = (t - t.mean()) / (t.std() + 1e-15)
        stat = _kstest(t, "norm").statistic
        if stat < best_ks:
            best_ks, best_theta = stat, theta
    return best_theta


# Columns that are guaranteed complete after merge_ud_vars:
#   • CY variables come from miceforest-imputed parquets (nb 01) — no NaN
#   • UD auxiliary variables are either zero-filled or conttype-filled
UD_PREDICTORS = [
    "year",
    "cinc_A", "cinc_B",
    "polity2_A", "polity2_B",
    "lpop_A", "lpop_B",
    "lgdp_A", "lgdp_B",
    "oil_A", "oil_B",
    "lmtnest_A", "lmtnest_B",
    "ncontig_A", "ncontig_B",
    "ud_conttype",
    "ud_iswar_allied", "ud_iswar_enemy", "ud_iswarin10yr",
    "ud_defense", "ud_nonaggression", "ud_entente",
    "ud_icow_nclaims", "ud_icow_maxsal",
]

# Variables to impute (may have NaN after merge_ud_vars + LOCF)
UD_IMPUTE_VARS = [
    "ud_peace",           # 91 % missing — structural but imputed per user preference
    "ud_A_biImports",     # 16 % missing
    "ud_B_biImports",     # 16 % missing
    "doe_pr_win_A",       #  3 % missing (post-LOCF)
    "doe_pr_win_B",       #  3 % missing (post-LOCF)
    "ud_log_capdist",     # <1 % missing (unknown country pairs)
]


def impute_ud_vars(
    ud: pd.DataFrame,
    n_imp: int = 5,
    random_state: int = 42,
) -> list[pd.DataFrame]:
    """
    Impute missing UD variables using miceforest (MICE + LightGBM).

    Trade flows are asinh-transformed before imputation and back-transformed
    afterward.  doe_pr_stalemate is derived post-imputation as
    max(0, 1 − pr_win_A − pr_win_B).

    Returns a list of n_imp imputed DataFrames (copies of ud).
    """
    # ── Asinh-transform trade flows ──────────────────────────────────────────
    thetas: dict[str, float] = {}
    ud_t = ud.reset_index(drop=True).copy()   # miceforest requires 0-based RangeIndex
    for var in ["ud_A_biImports", "ud_B_biImports"]:
        theta = _find_asinh_theta(ud_t[var])
        thetas[var] = theta
        ud_t[var] = np.arcsinh(theta * ud_t[var].clip(lower=0))

    # ── Build imputation frame: predictors + targets ──────────────────────────
    preds = [c for c in UD_PREDICTORS if c in ud_t.columns]
    imp_df = ud_t[preds + UD_IMPUTE_VARS].copy()

    # ── Fit miceforest kernel (v6: num_datasets, not datasets) ───────────────
    kernel = mf.ImputationKernel(
        imp_df,
        variable_schema=UD_IMPUTE_VARS,
        num_datasets=n_imp,
        random_state=random_state,
    )
    kernel.mice(iterations=2)

    # ── Extract n_imp imputed copies ─────────────────────────────────────────
    results: list[pd.DataFrame] = []
    for i in range(n_imp):
        ud_out = ud_t.copy()   # aligned to reset index
        completed = kernel.complete_data(dataset=i)

        for var in UD_IMPUTE_VARS:
            if var in completed.columns:
                ud_out[var] = completed[var].values

        # Inverse-transform trade
        for var in ["ud_A_biImports", "ud_B_biImports"]:
            ud_out[var] = (np.sinh(ud_out[var]) / thetas[var]).clip(lower=0)

        # Clip probabilistic variables to [0, 1]
        for var in ["ud_peace", "doe_pr_win_A", "doe_pr_win_B"]:
            ud_out[var] = ud_out[var].clip(0.0, 1.0)

        # Renormalise DOE: ensure pr_win_A + pr_stalemate + pr_win_B = 1
        doe_sum = ud_out["doe_pr_win_A"] + ud_out["doe_pr_win_B"]
        over = doe_sum > 1.0
        ud_out.loc[over, "doe_pr_win_A"] /= doe_sum[over]
        ud_out.loc[over, "doe_pr_win_B"] /= doe_sum[over]
        ud_out["doe_pr_stalemate"] = (
            1.0 - ud_out["doe_pr_win_A"] - ud_out["doe_pr_win_B"]
        ).clip(0.0, 1.0)

        results.append(ud_out)

    return results


print("UD imputation functions defined.")


UD imputation functions defined.


## § 12 — Main Loop: Build 25 DD Files (5 CY × 5 UD imputations)

For each of the 5 country-year imputed datasets (from notebook 01), we:

1. Cross-join states within each year to form the undirected dyad (UD) frame (~599k rows)
2. Merge all auxiliary variables
3. Run miceforest on the UD frame to produce 5 imputed UD copies  
   — imputation at the UD level guarantees $x_{AB} = x_{BA}$ for each copy
4. Expand each imputed UD copy to a directed dyad (DD) frame via `swap_ab`

Output: `dd_{cy}_{ud}.parquet`, where `cy` ∈ 1–5 and `ud` ∈ 1–5 (25 files total).


In [15]:

N_IMP_CY = 5   # CY-level imputations (from notebook 01)
N_IMP_UD = 5   # UD-level imputations (this notebook)

# Remove stale single-index files from a previous run
for i in range(1, N_IMP_CY + 1):
    p = INTERIM_OUT / f"dd_{i}.parquet"
    if p.exists():
        p.unlink()

for i_cy in range(1, N_IMP_CY + 1):
    print(f"\n── CY imputation {i_cy}/{N_IMP_CY} ──")

    print("  Building undirected dyads ...", end=" ", flush=True)
    ud = build_ud(INTERIM / f"cy_imputed_{i_cy}.parquet")
    print(f"{len(ud):,} UD rows")

    print("  Merging auxiliary variables ...", end=" ", flush=True)
    ud = merge_ud_vars(ud)
    print("done")

    print(f"  Imputing UD variables ({N_IMP_UD} datasets) ...", end=" ", flush=True)
    ud_imps = impute_ud_vars(ud, n_imp=N_IMP_UD, random_state=100 * i_cy)
    print("done")

    for i_ud, ud_imp in enumerate(ud_imps, start=1):
        out_path = INTERIM_OUT / f"dd_{i_cy}_{i_ud}.parquet"
        dd = ud_to_dd(ud_imp)
        dd = dd.drop(columns=["udyear"], errors="ignore")
        dd.to_parquet(out_path, index=False)
        print(f"  → {out_path.name}  ({len(dd):,} rows, {dd.shape[1]} cols)")

print(f"\nAll {N_IMP_CY * N_IMP_UD} DD files written to {INTERIM_OUT}.")



── CY imputation 1/5 ──
  Building undirected dyads ... 

605,866 UD rows
  Merging auxiliary variables ... 

done
  Imputing UD variables (5 datasets) ... 

done


  → dd_1_1.parquet  (1,211,732 rows, 117 cols)


...

  → dd_5_1.parquet  (1,211,732 rows, 117 cols)


  → dd_5_2.parquet  (1,211,732 rows, 117 cols)


  → dd_5_3.parquet  (1,211,732 rows, 117 cols)


  → dd_5_4.parquet  (1,211,732 rows, 117 cols)


  → dd_5_5.parquet  (1,211,732 rows, 117 cols)

All 25 DD files written to ../data/interim/lag1.


## ✓ Validation

In [16]:

import glob as _glob

# ── File inventory ────────────────────────────────────────────────────────────
dd_files = sorted(_glob.glob(str(INTERIM_OUT / "dd_[1-5]_[1-5].parquet")))
print(f"DD files produced: {len(dd_files)}  (expected 25)")

# Spot-check on dd_1_1
dd1 = pd.read_parquet(INTERIM_OUT / "dd_1_1.parquet")
print(f"\nShape (dd_1_1): {dd1.shape}")
print(f"Year range: {dd1['year'].min()}–{dd1['year'].max()}")

# ── Uniqueness ────────────────────────────────────────────────────────────────
n_dup = dd1["ddyear"].duplicated().sum()
msg = "✓" if n_dup == 0 else f"FAIL: {n_dup} duplicates"
print(f"\nDuplicate ddyear keys: {n_dup}  {msg}")

# ── Symmetry: every A→B row should have a matching B→A row ───────────────────
fwd = dd1.set_index(["ccode_A", "ccode_B", "year"]).index
rev = dd1.set_index(["ccode_B", "ccode_A", "year"]).index
n_asym = (~fwd.isin(rev)).sum()
msg = "✓" if n_asym == 0 else f"FAIL: {n_asym} asymmetric rows"
print(f"Symmetric coverage (A→B ↔ B→A): {msg}")

# ── t-1 shift spot-check (2026-07 rebuild) ────────────────────────────────────
# polity2_A on a year-t dyad row must equal the CY imputed polity2 at t-1.
_ci = pd.read_parquet(INTERIM / "cy_imputed_1.parquet",
                      columns=["ccode", "year", "polity2"])
_chk = (dd1[dd1["year"] == 1990][["ccode_A", "polity2_A"]]
        .drop_duplicates("ccode_A")
        .merge(_ci[_ci["year"] == 1989].rename(columns={"ccode": "ccode_A"}),
               on="ccode_A"))
_dev = (_chk["polity2_A"] - _chk["polity2"]).abs().max()
msg = "✓" if _dev < 1e-9 else f"FAIL: max dev {_dev}"
print(f"t-1 shift (polity2_A@1990 == imputed polity2@1989): {msg}")
assert "ongoing_wars_A" not in dd1.columns, "ongoing_wars leaked into dd!"
assert "other_ongoing_wars_A" in dd1.columns
print(f"other_ongoing_wars_A present; ongoing_wars_A absent ✓")

# ── log_capratio ─────────────────────────────────────────────────────────────
lr_sum = dd1["log_capratio"].sum()
print(f"log_capratio sum (≈0 expected): {lr_sum:.2f}")

# ── DOE ───────────────────────────────────────────────────────────────────────
doe_cov = dd1["doe_pr_win_A"].notna().mean()
doe_obs = dd1.dropna(subset=["doe_pr_win_A"])
doe_sum = doe_obs[["doe_pr_win_A", "doe_pr_stalemate", "doe_pr_win_B"]].sum(axis=1)
print(f"DOE pr_win_A coverage: {doe_cov:.1%}")
print(f"DOE probs sum to 1 (max dev): {(doe_sum - 1).abs().max():.4f}")

# ── Trade ─────────────────────────────────────────────────────────────────────
trade_cov = dd1["ud_A_biImports"].notna().mean()
print(f"Trade coverage (ud_A_biImports): {trade_cov:.1%}  (should be 100% after imputation)")

# ── Peace ─────────────────────────────────────────────────────────────────────
peace_cov = dd1["ud_peace"].notna().mean()
print(f"Peace score coverage: {peace_cov:.1%}  (should be 100% after imputation)")

# ── UD imputation produced variance ──────────────────────────────────────────
was_missing = pd.read_parquet(INTERIM_OUT / "dd_1_1.parquet")[["ddyear", "ud_A_biImports"]]
for i_ud in range(2, 6):
    alt = pd.read_parquet(INTERIM_OUT / f"dd_1_{i_ud}.parquet")[["ddyear", "ud_A_biImports"]]
    was_missing = was_missing.merge(alt, on="ddyear", suffixes=("", f"_{i_ud}"))
trade_cols = [c for c in was_missing.columns if c.startswith("ud_A_biImports")]
row_std = was_missing[trade_cols].std(axis=1)
n_varied = (row_std > 0).sum()
print(f"\nRows with imputation variance across UD datasets: {n_varied:,} / {len(dd1):,}")

# ── Alliances ─────────────────────────────────────────────────────────────────
n_def = (dd1["ud_defense"] > 0).sum()
print(f"Defense alliance dyad-years: {n_def:,} ({n_def/len(dd1):.1%})")

# ── Colonial flags ────────────────────────────────────────────────────────────
print(f"A_wasColOf_B=1: {(dd1['A_wasColOf_B']==1).sum():,}")
print(f"B_wasColOf_A=1: {(dd1['B_wasColOf_A']==1).sum():,}")

# ── Regan-period ──────────────────────────────────────────────────────────────
regan_dd = dd1[dd1["regan_period"]]
print(f"\nRegan-period rows (1946–1999): {len(regan_dd):,}")
print(f"B-country onset dyad-years:    {(regan_dd['onset_B']==1).sum():,}")

print("\n✓ Validation complete.")


DD files produced: 25  (expected 25)

Shape (dd_1_1): (1211732, 117)
Year range: 1946–2014

Duplicate ddyear keys: 0  ✓


Symmetric coverage (A→B ↔ B→A): ✓
t-1 shift (polity2_A@1990 == imputed polity2@1989): ✓
other_ongoing_wars_A present; ongoing_wars_A absent ✓
log_capratio sum (≈0 expected): -0.00
DOE pr_win_A coverage: 100.0%
DOE probs sum to 1 (max dev): 0.0000
Trade coverage (ud_A_biImports): 100.0%  (should be 100% after imputation)
Peace score coverage: 100.0%  (should be 100% after imputation)



Rows with imputation variance across UD datasets: 253,550 / 1,211,732
Defense alliance dyad-years: 71,846 (5.9%)
A_wasColOf_B=1: 6,513
B_wasColOf_A=1: 6,513

Regan-period rows (1946–1999): 831,392
B-country onset dyad-years:    19,839

✓ Validation complete.
